In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [2]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по птице v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Птица
50,АКМОЛИНСКАЯ ОБЛАСТЬ,2019-03-01,4332.93
453,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2015-12-01,4343.45
1872,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2025-01-01,414.05
96,АКМОЛИНСКАЯ ОБЛАСТЬ,2023-01-01,8549.71
828,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2016-04-01,57.04
26,АКМОЛИНСКАЯ ОБЛАСТЬ,2017-03-01,1835.41
1039,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2023-04-01,1097.14
772,ГШЫМКЕНТ,2022-03-01,24.06
471,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2017-06-01,4139.13
110,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-03-01,10969.85


In [3]:
# === загружаем данные ===
best_methods = pd.read_excel("results/Птица - Лучшие модели (MAPE_then_MAE) v2.xlsx")  # лучшие методы
best_methods

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АЛМАТИНСКАЯ ОБЛАСТЬ,8.81,1044.20,9.950000e+00,1202.61,10.37,1211.75,HW,MAPE,8.810000e+00,1044.20
1,ЖАМБЫЛСКАЯ ОБЛАСТЬ,25.54,511.77,2.821000e+01,552.49,32.70,643.10,HW,MAPE,2.554000e+01,511.77
2,КАРАГАНДИНСКАЯ ОБЛАСТЬ,8.14,80.45,1.498000e+01,152.15,9.47,93.85,HW,MAPE,8.140000e+00,80.45
3,ОБЛАСТЬ АБАЙ,7.37,149.35,7.450000e+00,156.44,11.73,228.58,HW,MAPE,7.370000e+00,149.35
4,ОБЛАСТЬ ЖЕТІСУ,121.22,65.18,6.138700e+02,364.69,275.58,170.66,HW,MAPE,1.212200e+02,65.18
5,ПАВЛОДАРСКАЯ ОБЛАСТЬ,38.39,157.48,4.050000e+01,157.96,60.60,213.54,HW,MAPE,3.839000e+01,157.48
6,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,44.78,256.75,4.731000e+01,281.66,52.90,314.60,HW,MAPE,4.478000e+01,256.75
7,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,24.55,172.47,3.317000e+01,239.15,31.45,232.66,HW,MAPE,2.455000e+01,172.47
8,АКТЮБИНСКАЯ ОБЛАСТЬ,277.84,26.98,2.852000e+02,31.75,143.70,28.84,Prophet,MAPE,1.437000e+02,26.98
9,ГАЛМАТЫ,NaN,0.11,4.612450e+03,4.61,99.82,0.10,Prophet,MAPE,9.982000e+01,0.10


In [4]:
actual_aug = pd.read_excel("Птица 08.2025.xlsx")
actual_aug["Период"] = pd.to_datetime(actual_aug["Период"], format="%Y-%m")
actual_aug["Птица"] = (actual_aug["Птица"]
                     .astype(str)
                     .str.replace(".", "", regex=False)   # убираем разделители тысяч
                     .str.replace(",", ".", regex=False)  # заменяем запятую на точку
                     .astype(float))
actual_aug.to_excel("Птица обработанные август 2025.xlsx", index=False)
actual_aug



,Регион,Период,Птица
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2025-08-01,10211.00
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2025-08-01,20.40
2,АЛМАТИНСКАЯ ОБЛАСТЬ,2025-08-01,10015.99
3,АТЫРАУСКАЯ ОБЛАСТЬ,2025-08-01,NaN
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2025-08-01,942.85
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2025-08-01,1661.78
6,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2025-08-01,1047.46
7,КОСТАНАЙСКАЯ ОБЛАСТЬ,2025-08-01,1027.62
8,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2025-08-01,12.15
9,МАНГИСТАУСКАЯ ОБЛАСТЬ,2025-08-01,1014.60


In [5]:
# === настройки ===
TARGET = "Птица"
CUTOFF = "2025-07-01"
FORECAST = "2025-08-01"
EPS = 1e-6
SEAS = 12

In [6]:
# оставляем только август 2025 для проверки
fact_aug = (actual_aug[actual_aug["Период"] == "2025-08-01"]
            .set_index("Регион")[TARGET])


In [7]:
# === функции прогнозов, строго как в обучающем коде ===
def fc_hw_like_training(train):
    # train — Series с MS частотой
    train_log = np.log1p(train)  # log1p
    model = ExponentialSmoothing(train_log, seasonal="add", seasonal_periods=SEAS)\
            .fit(optimized=True)
    fc_log = model.forecast(1)
    return float(np.expm1(fc_log).iloc[0])  # expm1

In [8]:
def fc_sarima_like_training(train):
    train_plus = train + EPS
    train_log  = np.log(train_plus)
    use_seasonal = len(train_log) >= 2 * SEAS

    sar = auto_arima(
        train_log,
        seasonal=use_seasonal,
        m=SEAS if use_seasonal else 1,
        D=1 if use_seasonal else 0,
        seasonal_test=None,
        boxcox=True,            # как в обучении
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore"
    )

    fc_log = sar.predict(n_periods=1)
    # берём первый элемент позиционно, независимо от типа (Series/ndarray/scalar)
    fc_log_scalar = np.asarray(fc_log).ravel()[0]

    return float(np.exp(fc_log_scalar) - EPS)

In [9]:
def fc_prophet_like_training(train):
    df_p = (train.reset_index()
                 .rename(columns={"Период": "ds", TARGET: "y"}))
    df_p["y"] = np.log(df_p["y"] + EPS)       # лог как в обучении
    m = Prophet()
    m.fit(df_p)
    future = m.make_future_dataframe(periods=1, freq="MS")
    yhat_log = m.predict(future)["yhat"].iloc[-1]
    return float(np.exp(yhat_log) - EPS)

In [10]:
methods_map = {
    "HW": fc_hw_like_training,
    "Holt-Winters": fc_hw_like_training,
    "Holt_Winters": fc_hw_like_training,
    "SARIMA": fc_sarima_like_training,
    "Prophet": fc_prophet_like_training,
}

In [11]:
# === прогон по регионам согласно «лучшему методу» ===
rows = []
for _, r in best_methods.iterrows():
    region = r["Регион"]
    method = r["Best_method"]

    ts = (df[df["Регион"] == region]
          .set_index("Период")[TARGET]
          .asfreq("MS")
          .sort_index())

    train = ts[:CUTOFF].dropna()
    if len(train) < 24:
        # как и в обучении, пропускаем короткие ряды
        continue

    # вызов нужной функции
    f = methods_map.get(method)
    if f is None:
        # на всякий случай нормализуем ключи
        key = str(method).strip().upper()
        if key == "HW" or "HOLT" in key:
            f = fc_hw_like_training
        elif "SARIMA" in key or "ARIMA" in key:
            f = fc_sarima_like_training
        else:
            f = fc_prophet_like_training

    fc = f(train)
    actual = fact_aug.get(region, np.nan)
    pct_dev = (fc - actual) / actual * 100 if pd.notna(actual) else np.nan

    rows.append({
        "Регион": region,
        "Лучший метод": method,
        "Прогноз (2025-08)": round(fc, 2),
        "Факт (2025-08)": round(actual, 2) if pd.notna(actual) else np.nan,
        "Отклонение, %": round(pct_dev, 2) if pd.notna(pct_dev) else np.nan
    })

results_aug = pd.DataFrame(rows).sort_values("Регион").reset_index(drop=True)
results_aug.to_excel("results/Птица - Прогноз на 2025-08 (как в обучении).xlsx")
results_aug

19:03:56 - cmdstanpy - INFO - Chain [1] start processing
19:03:56 - cmdstanpy - INFO - Chain [1] done processing
19:03:56 - cmdstanpy - INFO - Chain [1] start processing
19:03:56 - cmdstanpy - INFO - Chain [1] done processing
19:03:56 - cmdstanpy - INFO - Chain [1] start processing
19:03:56 - cmdstanpy - INFO - Chain [1] done processing


,Регион,Лучший метод,Прогноз (2025-08),Факт (2025-08),"Отклонение, %"
0,АКМОЛИНСКАЯ ОБЛАСТЬ,SARIMA,10748.37,10211.00,5.26
1,АКТЮБИНСКАЯ ОБЛАСТЬ,Prophet,27.21,20.40,33.39
2,АЛМАТИНСКАЯ ОБЛАСТЬ,HW,9973.02,10015.99,-0.43
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,SARIMA,6274.98,5862.00,7.05
4,ГАЛМАТЫ,Prophet,0.00,NaN,NaN
5,ГАСТАНА,SARIMA,-0.00,NaN,NaN
6,ГШЫМКЕНТ,SARIMA,3.96,NaN,NaN
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,HW,1330.08,1661.78,-19.96
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,SARIMA,964.29,942.85,2.27
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,HW,885.14,1047.46,-15.50
